```{contents}
:local:
:depth: 2
```

# Problems: Linear Regression

:::{admonition} Before you start
:class: tip

This problem set accompanies {doc}`Linear Regression </1-numerical_methods/Topic1.3-Linear_Regression>`. It is worth **100 points**:

- **Part A — Skill Checks (30 pts)** — short answers, auto-graded, resubmit as often as
  you like until they pass.
- **Part B — Visualization (35 pts)** — plots plus written interpretation, peer graded.
- **Part C — Open Ended (35 pts)** — one synthesis problem, peer graded.

The parts build on each other: Part A works out the syntax you need for Part B, and
Part B produces the evidence you argue from in Part C. Do them in order.
:::

## Setup

The chapter fit basis functions to an **ethanol** infrared spectrum. Here you will work
with a **methanol** spectrum measured under different conditions: gas phase, 70 mmHg of
methanol in nitrogen to 600 mmHg total, 5 cm path length, 2 cm⁻¹ resolution. It comes
from the Coblentz Society collection in the NIST Chemistry WebBook (spectrum 8791,
measured at Dow Chemical in 1964), converted from transmittance to absorbance by
$A = -\log_{10} T$.

Methanol is the simplest alcohol, so its spectrum is a stripped-down version of
ethanol's: a strong C–O stretch near 1030 cm⁻¹ and a cluster of C–H stretches between
2800 and 3050 cm⁻¹, without ethanol's extra methylene modes. That makes it a good place
to ask how many basis functions a band actually needs.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    plt.style.use('../settings/plot_style.mplstyle')   # available inside the book
except OSError:
    pass                                              # downloaded notebook: use defaults

df = pd.read_csv('data/methanol_IR.csv')
x_all = df['wavenumber [cm^-1]'].values
y_all = df['absorbance'].values
print(f"{len(x_all)} points, {x_all.min():.1f}-{x_all.max():.1f} cm^-1")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_all, y_all, lw=0.8)
ax.set_xlabel('wavenumber [cm$^{-1}$]')
ax.set_ylabel('absorbance');

Run this once to load the autograder. It will not run inside the book — use the
downloaded notebook.

In [ ]:
import otter
grader = otter.Notebook()

---

## Part A — Skill Checks (30 pts)

Three questions, 10 points each. Each asks you to assign a single number to a named
variable. Run the check cell after each one; there is no limit on attempts.

### A1. Locate the C–O stretch (10 pts)

:::{exercise}
:label: pr-nm-meoh-co-peak

Restrict the spectrum to the window $950 \le \tilde\nu \le 1120$ cm⁻¹, which contains the
C–O stretch and its two shoulders. Find the wavenumber at which absorbance is largest in
that window.

Assign the wavenumber in cm⁻¹ to `peak_wavenumber`.
:::

In [ ]:
# YOUR CODE HERE
peak_wavenumber = ...

In [ ]:
grader.check("q1")

### A2. Fit one Gaussian to the band (10 pts)

:::{exercise}
:label: pr-nm-meoh-gauss-fit

Model the same window as a single Gaussian sitting on a flat baseline:

$$
A(\tilde\nu) \;=\; w_0 \exp\!\left[-\frac{(\tilde\nu - c)^2}{2\sigma^2}\right] + w_1
$$

with the center $c$ fixed at your answer from A1 and $\sigma = 25$ cm⁻¹ (the same width
the chapter used). Build the two-column design matrix $\bar{\bar{X}}$ — one column for the
Gaussian, one of ones for the baseline — and solve the normal equations
$\bar{\bar{X}}^T\bar{\bar{X}}\vec{w} = \bar{\bar{X}}^T\vec{y}$ with `np.linalg.solve`.

Assign the Gaussian coefficient $w_0$ to `w_gauss`.
:::

In [ ]:
# YOUR CODE HERE
w_gauss = ...

In [ ]:
grader.check("q2")

### A3. Score the fit (10 pts)

:::{exercise}
:label: pr-nm-meoh-fit-r2

Compute the coefficient of determination for the A2 fit **over the same window**:

$$
r^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

Assign it to `r2_single`.
:::

In [ ]:
# YOUR CODE HERE
r2_single = ...

In [ ]:
grader.check("q3")

---

## Part B — Visualization (35 pts)

:::{exercise}
:label: pr-nm-meoh-ch-basis

A single Gaussian was not enough. Now work on the **C–H stretch region**,
$2800 \le \tilde\nu \le 3050$ cm⁻¹, and let the number of basis functions vary.

Reuse the chapter's `gaussian_features(x, N, sigma=25)` idea: place `N` Gaussian centers
evenly across the region, append a column of ones for the baseline, and solve the normal
equations exactly as you did in A2.

1. For `N = 1, 2, 4, 6`, fit the region and plot the data with all four fits overlaid on
   one set of axes. Label each curve with its `N`.
2. Below that, plot the **residuals** ($y - \hat{y}$) for `N = 1` and `N = 6` against
   wavenumber, on shared x-axes with the panel above.
3. Compute $r^2$ for each `N` and report the four values.
4. In **3–5 sentences**: the jump in $r^2$ between `N = 2` and `N = 4` is much larger than
   the jump between `N = 2` and `N = 3`. Explain what feature of the underlying
   spectroscopy that pattern reflects, and say what the `N = 1` residuals show that the
   $r^2$ value alone does not.

Label your axes with units.
:::

In [ ]:
# YOUR CODE HERE
fig, ax = plt.subplots()

---

## Part C — Open Ended (35 pts)

:::{exercise}
:label: pr-nm-meoh-basis-cond

The chapter presented two ways to build a design matrix: **polynomial** (Vandermonde)
features and **Gaussian** features. Part B used Gaussians. Decide which basis you would
actually recommend for fitting the C–H region, and defend it with evidence.

Your answer should include:

1. Polynomial fits of the same $2800$–$3050$ cm⁻¹ region at several orders (go at least
   as high as order 10), with $r^2$ for each.
2. For every model you fit — polynomial and Gaussian — the condition number of
   $\bar{\bar{X}}^T\bar{\bar{X}}$, reported alongside its $r^2$. Recall from
   {doc}`Linear Algebra </1-numerical_methods/Topic1.2-Linear_Algebra>` what a large condition number implies about solving a
   linear system.
3. A short written argument (**one paragraph**) for which basis you would use and why,
   citing your own numbers.

Something surprising happens to the high-order polynomial fits. Do not just report the
$r^2$ — explain it.

There is more than one defensible recommendation here. You are graded on the reasoning
and the evidence, not on reaching a particular verdict.
:::

In [ ]:
# YOUR CODE HERE

---

## Summary

- Part A built a one-column Gaussian design matrix and solved the normal equations by
  hand, reproducing the chapter's workflow on a new spectrum. A single Gaussian recovers
  only $r^2 = 0.65$ on the C–O band.
- Part B let the basis size vary across the C–H stretch region and showed that fit quality
  improves sharply once the number of basis functions matches the number of physical
  sub-bands — and that residual *structure* diagnoses a wrong model form in a way a
  scalar $r^2$ cannot.
- Part C set the Gaussian basis against a polynomial one and asked you to weigh fit
  quality against numerical conditioning, connecting this chapter back to the condition
  number from {doc}`Linear Algebra </1-numerical_methods/Topic1.2-Linear_Algebra>`.

## Additional Reading

1. NIST Chemistry WebBook, SRD 69 — [methanol IR spectrum](https://webbook.nist.gov/cgi/cbook.cgi?ID=C67561&Type=IR-SPEC),
   Coblentz Society collection no. 8791.
2. [`numpy.linalg.cond`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.cond.html)
   and [`numpy.polynomial`](https://numpy.org/doc/stable/reference/routines.polynomials.html)
   for well-conditioned polynomial bases.